# Teammate Processed Data and Notebook Audit

This notebook audits the processed outputs and notebooks for sections 2.1, 2.2, and 2.3 without modifying teammate files.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    if start.name == "notebooks":
        start = start.parent
    for candidate in [start] + list(start.parents):
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUT_DIR = PROJECT_ROOT / "outputs" / "teammate_audit"
FIG_DIR = OUT_DIR / "figures"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUT_DIR:", OUT_DIR)

def safe_read_csv(path: Path, **kwargs) -> pd.DataFrame:
    """Read a CSV safely; return an empty DataFrame if the optional CSV is empty/missing."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        print(f"Optional CSV missing or empty: {path}")
        return pd.DataFrame()
    try:
        return pd.read_csv(path, **kwargs)
    except pd.errors.EmptyDataError:
        print(f"Optional CSV has no columns: {path}")
        return pd.DataFrame()


PROJECT_ROOT: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project
OUT_DIR: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/teammate_audit


In [2]:
from src.teammate_audit import (
    build_paths,
    run_full_teammate_audit,
)

## 1. Run Audit

This cell runs the full audit. It reads CSVs fully and samples large parquet market data files.

In [3]:
results = run_full_teammate_audit(PROJECT_ROOT)
print("Audit complete")
print("Report:", OUT_DIR / "final_teammate_audit_report.md")

/Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/src/teammate_audit.py:229: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = df.assign(_date=dates).groupby([stock_col, "_date"], sort=False).size()
/Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/src/teammate_audit.py:262: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  gaps = temp.groupby([stock_col, "_date"], sort=False)["_timestamp"].diff().dt.total_seconds()
/Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/src/teamma

Audit complete
Report: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/teammate_audit/final_teammate_audit_report.md


## 2. File Inventory

In [4]:
inventory = safe_read_csv(OUT_DIR / "file_inventory.csv")
display(inventory.head(20))
if {"folder_label", "suffix", "relative_path", "size_mb"}.issubset(inventory.columns):
    display(inventory.groupby(["folder_label", "suffix"]).agg(n_files=("relative_path", "size"), size_mb=("size_mb", "sum")).reset_index())
    display(inventory.sort_values("size_mb", ascending=False).head(15))
else:
    print("Inventory table is empty or missing expected columns.")

,relative_path,folder_label,file_name,suffix,size_mb,modified_time,parent_folder
0,data/processed_2_1/baseline_20stocks/baseline_...,processed_2_1,baseline_20stocks_summary.csv,.csv,0.003497,2026-05-14 09:46:56.375961065,data/processed_2_1/baseline_20stocks
1,data/processed_2_1/baseline_20stocks/bin_test_...,processed_2_1,bin_test_201902_20stocks.parquet,.parquet,48.266534,2026-05-14 09:46:56.418492317,data/processed_2_1/baseline_20stocks
2,data/processed_2_1/baseline_20stocks/bin_train...,processed_2_1,bin_train_201901_20stocks.parquet,.parquet,53.306490,2026-05-14 09:46:56.465441227,data/processed_2_1/baseline_20stocks
3,data/processed_2_1/baseline_20stocks/fills_tes...,processed_2_1,fills_test_201902_available_baseline_stocks.pa...,.parquet,16.432052,2026-05-14 09:46:56.479948044,data/processed_2_1/baseline_20stocks
4,data/processed_2_1/baseline_20stocks/fills_tra...,processed_2_1,fills_train_201901_available_baseline_stocks.p...,.parquet,25.751740,2026-05-14 09:46:56.498739243,data/processed_2_1/baseline_20stocks
5,data/processed_2_1/baseline_20stocks/section_2...,processed_2_1,section_2_1_baseline_config.json,.json,0.001097,2026-05-14 09:46:56.499426842,data/processed_2_1/baseline_20stocks
6,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201901_standardised.parquet,.parquet,110.143414,2026-05-14 09:46:56.587696791,data/processed_2_1/monthly_standardised/bin
7,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201901_stock_stats.csv,.csv,0.013821,2026-05-14 09:46:56.589473009,data/processed_2_1/monthly_standardised/bin
8,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201902_standardised.parquet,.parquet,99.729477,2026-05-14 09:46:56.670475721,data/processed_2_1/monthly_standardised/bin
9,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201902_stock_stats.csv,.csv,0.013814,2026-05-14 09:46:56.672334433,data/processed_2_1/monthly_standardised/bin


,folder_label,suffix,n_files,size_mb
0,processed_2_1,.csv,34,0.387774
1,processed_2_1,.json,2,0.002006
2,processed_2_1,.parquet,28,1739.937391
3,processed_2_2_rolling_baseline,.csv,51,1.086388
4,processed_2_2_rolling_baseline,.json,1,0.001968
5,processed_2_2_rolling_baseline,.png,3,0.144160


,relative_path,folder_label,file_name,suffix,size_mb,modified_time,parent_folder
14,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201905_standardised.parquet,.parquet,123.107119,2026-05-14 09:46:57.155452251,data/processed_2_1/monthly_standardised/bin
20,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201908_standardised.parquet,.parquet,123.008862,2026-05-14 09:46:57.562015772,data/processed_2_1/monthly_standardised/bin
24,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201910_standardised.parquet,.parquet,116.589676,2026-05-14 09:46:57.774447680,data/processed_2_1/monthly_standardised/bin
10,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201903_standardised.parquet,.parquet,115.285206,2026-05-14 09:46:56.856461287,data/processed_2_1/monthly_standardised/bin
18,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201907_standardised.parquet,.parquet,113.451562,2026-05-14 09:46:57.444001675,data/processed_2_1/monthly_standardised/bin
6,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201901_standardised.parquet,.parquet,110.143414,2026-05-14 09:46:56.587696791,data/processed_2_1/monthly_standardised/bin
12,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201904_standardised.parquet,.parquet,108.509168,2026-05-14 09:46:56.987235308,data/processed_2_1/monthly_standardised/bin
16,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201906_standardised.parquet,.parquet,106.278826,2026-05-14 09:46:57.292462587,data/processed_2_1/monthly_standardised/bin
22,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201909_standardised.parquet,.parquet,102.498122,2026-05-14 09:46:57.652247190,data/processed_2_1/monthly_standardised/bin
8,data/processed_2_1/monthly_standardised/bin/bi...,processed_2_1,bin_201902_standardised.parquet,.parquet,99.729477,2026-05-14 09:46:56.670475721,data/processed_2_1/monthly_standardised/bin


## 3. Processed Data Schemas

In [5]:
schema = safe_read_csv(OUT_DIR / "csv_schema_summary.csv")
coverage = safe_read_csv(OUT_DIR / "csv_key_coverage_summary.csv")
missing = safe_read_csv(OUT_DIR / "csv_missing_summary.csv")
numeric = safe_read_csv(OUT_DIR / "csv_numeric_summary.csv")
display(schema.head(20))
display(coverage.head(20))
display(missing.sort_values("missing_pct", ascending=False).head(30))

,source_file,suffix,n_rows,n_cols,columns,dtypes,memory_mb,sampled,error
0,data/processed_2_1/baseline_20stocks/baseline_...,.csv,40.0,10.0,month|stock|n_rows|n_trading_days|total_abs_tr...,"{""month"": ""int64"", ""stock"": ""object"", ""n_rows""...",0.008957,False,NaN
1,data/processed_2_1/baseline_20stocks/bin_test_...,.parquet,5000.0,34.0,date|time|stock|trade|orderFlow|hidden|auction...,"{""date"": ""string"", ""time"": ""string"", ""stock"": ...",2.204599,True,NaN
2,data/processed_2_1/baseline_20stocks/bin_train...,.parquet,5000.0,34.0,date|time|stock|trade|orderFlow|hidden|auction...,"{""date"": ""string"", ""time"": ""string"", ""stock"": ...",2.209480,True,NaN
3,data/processed_2_1/baseline_20stocks/fills_tes...,.parquet,5000.0,23.0,date|stock|time|trade|mid|spread|effSpread|dep...,"{""date"": ""string"", ""stock"": ""category"", ""time""...",1.890309,True,NaN
4,data/processed_2_1/baseline_20stocks/fills_tra...,.parquet,5000.0,23.0,date|stock|time|trade|mid|spread|effSpread|dep...,"{""date"": ""string"", ""stock"": ""category"", ""time""...",1.895190,True,NaN
5,data/processed_2_1/monthly_standardised/bin/bi...,.parquet,5000.0,34.0,date|time|stock|trade|orderFlow|hidden|auction...,"{""date"": ""string"", ""time"": ""string"", ""stock"": ...",2.233322,True,NaN
6,data/processed_2_1/monthly_standardised/bin/bi...,.csv,50.0,19.0,stock|first_datetime|last_datetime|n_rows|n_tr...,"{""stock"": ""object"", ""first_datetime"": ""object""...",0.029417,False,NaN
7,data/processed_2_1/monthly_standardised/bin/bi...,.parquet,5000.0,34.0,date|time|stock|trade|orderFlow|hidden|auction...,"{""date"": ""string"", ""time"": ""string"", ""stock"": ...",2.233210,True,NaN
8,data/processed_2_1/monthly_standardised/bin/bi...,.csv,50.0,19.0,stock|first_datetime|last_datetime|n_rows|n_tr...,"{""stock"": ""object"", ""first_datetime"": ""object""...",0.029417,False,NaN
9,data/processed_2_1/monthly_standardised/bin/bi...,.parquet,5000.0,34.0,date|time|stock|trade|orderFlow|hidden|auction...,"{""date"": ""string"", ""time"": ""string"", ""stock"": ...",2.233322,True,NaN


,source_file,n_rows,n_unique_stocks,n_unique_dates,min_date,max_date,n_unique_times,min_time,max_time,rows_per_stock_date_median,rows_per_stock_date_min,rows_per_stock_date_max
0,data/processed_2_1/baseline_20stocks/baseline_...,40,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,data/processed_2_1/baseline_20stocks/bin_test_...,5000,1.0,3.0,2019-02-01,2019-02-05,2340.0,09:30:00,15:59:50,0.0,0.0,2330.0
2,data/processed_2_1/baseline_20stocks/bin_train...,5000,1.0,3.0,2019-01-02,2019-01-04,2340.0,09:30:00,15:59:50,0.0,0.0,2339.0
3,data/processed_2_1/baseline_20stocks/fills_tes...,5000,1.0,1.0,2019-02-01,2019-02-01,4719.0,09:30:00.110,10:09:11.431,5000.0,5000.0,5000.0
4,data/processed_2_1/baseline_20stocks/fills_tra...,5000,1.0,1.0,2019-01-02,2019-01-02,4883.0,09:30:00.142,10:09:14.916,5000.0,5000.0,5000.0
5,data/processed_2_1/monthly_standardised/bin/bi...,5000,1.0,3.0,2019-01-02,2019-01-04,2338.0,09:30:00,15:59:50,0.0,0.0,2311.0
6,data/processed_2_1/monthly_standardised/bin/bi...,50,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,data/processed_2_1/monthly_standardised/bin/bi...,5000,1.0,3.0,2019-02-01,2019-02-05,2316.0,09:30:00,15:59:50,0.0,0.0,2095.0
8,data/processed_2_1/monthly_standardised/bin/bi...,50,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,data/processed_2_1/monthly_standardised/bin/bi...,5000,1.0,3.0,2019-03-01,2019-03-05,2327.0,09:30:00,15:59:50,0.0,0.0,2201.0


,source_file,column,missing_count,missing_pct
1616,data/processed_2_2_rolling_baseline/parameters...,x_hidden,546,100.00
1620,data/processed_2_2_rolling_baseline/parameters...,spread_bps,546,100.00
1619,data/processed_2_2_rolling_baseline/parameters...,effLobImb,546,100.00
1615,data/processed_2_2_rolling_baseline/parameters...,x_trade,546,100.00
1803,data/processed_2_2_rolling_baseline/parameters...,half_life_sec,546,100.00
1618,data/processed_2_2_rolling_baseline/parameters...,lobImb,546,100.00
1806,data/processed_2_2_rolling_baseline/parameters...,ow_state_pre,546,100.00
1617,data/processed_2_2_rolling_baseline/parameters...,x_flow_depth,546,100.00
346,data/processed_2_1/monthly_standardised/bin/bi...,effSpread,2918,58.36
682,data/processed_2_1/monthly_standardised/bin/bi...,eff_spread_bps,2918,58.36


## 4. Artifact Classification

In [6]:
artifacts = safe_read_csv(OUT_DIR / "artifact_classification.csv")
if {"likely_section", "artifact_type"}.issubset(artifacts.columns):
    display(artifacts.groupby(["likely_section", "artifact_type"]).size().reset_index(name="n"))
else:
    print("Artifact classification is empty or missing expected columns.")
display(artifacts.head(30))

,likely_section,artifact_type,n
0,2.1,cleaned_bins,24
1,2.1,cleaned_fills,24
2,2.1,selected_universe,8
3,2.1,unknown,6
4,2.2/2.3,rolling_windows,50
5,2.2/2.3,unknown,1


,artifact_type,likely_section,source_file,detected_columns,n_rows,date_range,stock_count,notes
0,selected_universe,2.1,data/processed_2_1/baseline_20stocks/baseline_...,month|stock|n_rows|n_trading_days|total_abs_tr...,40.0,NaT to NaT,20.0,NaN
1,selected_universe,2.1,data/processed_2_1/baseline_20stocks/bin_test_...,date|time|stock|trade|orderFlow|hidden|auction...,5000.0,2019-02-01 00:00:00 to 2019-02-05 00:00:00,1.0,NaN
2,selected_universe,2.1,data/processed_2_1/baseline_20stocks/bin_train...,date|time|stock|trade|orderFlow|hidden|auction...,5000.0,2019-01-02 00:00:00 to 2019-01-04 00:00:00,1.0,NaN
3,selected_universe,2.1,data/processed_2_1/baseline_20stocks/fills_tes...,date|stock|time|trade|mid|spread|effSpread|dep...,5000.0,2019-02-01 00:00:00 to 2019-02-01 00:00:00,1.0,NaN
4,selected_universe,2.1,data/processed_2_1/baseline_20stocks/fills_tra...,date|stock|time|trade|mid|spread|effSpread|dep...,5000.0,2019-01-02 00:00:00 to 2019-01-02 00:00:00,1.0,NaN
5,cleaned_bins,2.1,data/processed_2_1/monthly_standardised/bin/bi...,date|time|stock|trade|orderFlow|hidden|auction...,5000.0,2019-01-02 00:00:00 to 2019-01-04 00:00:00,1.0,NaN
6,cleaned_bins,2.1,data/processed_2_1/monthly_standardised/bin/bi...,stock|first_datetime|last_datetime|n_rows|n_tr...,50.0,NaT to NaT,50.0,NaN
7,cleaned_bins,2.1,data/processed_2_1/monthly_standardised/bin/bi...,date|time|stock|trade|orderFlow|hidden|auction...,5000.0,2019-02-01 00:00:00 to 2019-02-05 00:00:00,1.0,NaN
8,cleaned_bins,2.1,data/processed_2_1/monthly_standardised/bin/bi...,stock|first_datetime|last_datetime|n_rows|n_tr...,50.0,NaT to NaT,50.0,NaN
9,cleaned_bins,2.1,data/processed_2_1/monthly_standardised/bin/bi...,date|time|stock|trade|orderFlow|hidden|auction...,5000.0,2019-03-01 00:00:00 to 2019-03-05 00:00:00,1.0,NaN


## 5. Universe and Date Split Candidates

In [8]:
for name in ["selected_universe_candidates.csv", "date_split_candidates.csv", "rolling_window_candidates.csv"]:
    p = OUT_DIR / name
    print("", name)
    df = safe_read_csv(p) if p.exists() else pd.DataFrame()
    display(df.head(20))

 selected_universe_candidates.csv


,source_file,stock_count,stocks,exactly_20,date_coverage
0,data/processed_2_1/baseline_20stocks/baseline_...,20,"AAL,AAP,AAPL,ABBV,ABT,ACN,ADBE,ADI,ADP,ADSK,AG...",True,NaN
1,data/processed_2_1/monthly_standardised/fills/...,1,AAPL,False,NaN
2,data/processed_2_1/monthly_standardised/fills/...,1,AAPL,False,NaN
3,data/processed_2_1/monthly_standardised/fills/...,1,AAPL,False,NaN
4,data/processed_2_1/monthly_standardised/fills/...,1,AAPL,False,NaN
5,data/processed_2_1/monthly_standardised/fills/...,1,AAPL,False,NaN
6,data/processed_2_1/monthly_standardised/fills/...,1,AAPL,False,NaN
7,data/processed_2_1/monthly_standardised/fills/...,1,AAPL,False,NaN
8,data/processed_2_1/monthly_standardised/fills/...,1,AAPL,False,NaN
9,data/processed_2_1/monthly_standardised/fills/...,1,AAPL,False,NaN


 date_split_candidates.csv


,source_file,columns,n_rows,train_start,test_start,stock_count
0,data/processed_2_1/rolling_full_universe/rolli...,pair_id|train_month|test_month|train_bin_path|...,11,201901,201902,NaN
1,data/processed_2_1/rolling_full_universe/rolli...,stock|train_total_abs_notional|train_n_rows|tr...,546,201901,201902,50.0
2,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201901,201902,NaN
3,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201902,201903,NaN
4,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201903,201904,NaN
5,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201904,201905,NaN
6,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201905,201906,NaN
7,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201906,201907,NaN
8,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201907,201908,NaN
9,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201908,201909,NaN


 rolling_window_candidates.csv


,source_file,columns,n_rows,train_start,test_start,stock_count
0,data/processed_2_1/rolling_full_universe/rolli...,pair_id|train_month|test_month|train_bin_path|...,11,201901,201902,NaN
1,data/processed_2_1/rolling_full_universe/rolli...,stock|train_total_abs_notional|train_n_rows|tr...,546,201901,201902,50.0
2,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201901,201902,NaN
3,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201902,201903,NaN
4,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201903,201904,NaN
5,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201904,201905,NaN
6,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201905,201906,NaN
7,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201906,201907,NaN
8,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201907,201908,NaN
9,data/processed_2_2_rolling_baseline/metrics/ov...,pair_id|train_month|test_month|model|sample|n|...,4,201908,201909,NaN


## 6. Data Quality and Time Gaps

In [ ]:
quality = safe_read_csv(OUT_DIR / "data_quality_summary.csv")
time_gaps = safe_read_csv(OUT_DIR / "time_gap_summary.csv")
jumps = safe_read_csv(OUT_DIR / "large_price_jumps_processed.csv")
display(quality.head(30))
if "median_gap_seconds" in time_gaps.columns:
    display(time_gaps.sort_values("median_gap_seconds").head(20))
else:
    display(time_gaps.head(20))
display(jumps.head(20))

EmptyDataError: No columns to parse from file

## 7. Scaling Factor Audit

In [ ]:
scaling_audit = safe_read_csv(OUT_DIR / "scaling_factor_audit.csv")
scaling_cov = safe_read_csv(OUT_DIR / "scaling_coverage_by_date.csv") if (OUT_DIR / "scaling_coverage_by_date.csv").exists() else pd.DataFrame()
display(scaling_audit)
display(scaling_cov.head(20))

## 8. Notebook Audit

In [ ]:
nb_index = safe_read_csv(OUT_DIR / "notebook_cell_index.csv")
display(nb_index.head(50))
if "keyword_hits" in nb_index.columns:
    display(nb_index[nb_index["keyword_hits"].fillna("").str.len() > 0].head(50))
extracts = OUT_DIR / "notebook_code_extracts.md"
print(extracts.read_text()[:5000] if extracts.exists() else "No extracts")

## 9. Model Fit Audit

In [ ]:
model_audit = safe_read_csv(OUT_DIR / "model_fit_audit.csv")
param_files = safe_read_csv(OUT_DIR / "fitted_parameter_files.csv")
display(model_audit)
display(param_files.head(20))

## 10. Backtest Engine Audit

In [ ]:
backtest = safe_read_csv(OUT_DIR / "backtest_engine_audit.csv")
display(backtest)

## 11. Compatibility With My Modules

In [ ]:
compat = safe_read_csv(OUT_DIR / "compatibility_audit.csv")
todo = safe_read_csv(OUT_DIR / "integration_todo_list.csv")
display(compat)
display(todo)

## 12. Figures

In [ ]:
for p in sorted(FIG_DIR.glob("*.png")):
    print(p.relative_to(PROJECT_ROOT))

## 13. Final Report

In [ ]:
report_path = OUT_DIR / "final_teammate_audit_report.md"
print(report_path.read_text() if report_path.exists() else "Report missing")